# LangChain Chains and LCEL: Orchestrating Multi-Step AI Workflows

## 1. Theory & Real-World Context

A **Chain** is an orchestrated sequence of operations where the output of one component becomes the input of the next. While a single LLM call is useful for basic Q&A, enterprise generative AI applications require complex workflows: data retrieval, prompt formatting, LLM invocation, output parsing, and conditional routing.

**The Paradigm Shift: Legacy vs. LCEL**

* **Legacy Chains (`LLMChain`, `SequentialChain`):** LangChain originally used Python classes with heavy boilerplate to link steps. These classes (like the one in your provided example) are now formally **deprecated**. They are rigid, hard to debug, and struggle with modern requirements like asynchronous execution and real-time UI streaming.
* **LangChain Expression Language (LCEL):** The modern standard. LCEL uses the Unix-style pipe operator (`|`) to declare chains. It relies on a unified interface (`Runnable`). Because every component in an LCEL chain is a `Runnable`, you automatically get `.invoke()`, `.stream()`, `.batch()`, and `.astream()` for free, without rewriting any logic.

**Processing Large Datasets (Overcoming Context Limits)**
If you pass a 500-page test execution report to an LLM, it will crash with a `Token Limit Exceeded` error. Chains solve this using **Map-Reduce**:

1. **Map:** Chunk the document. Pass each chunk through a chain to generate a mini-summary.
2. **Reduce:** Chain those mini-summaries together and pass them to a final LLM call to produce the master summary.

> **Real-World AI Engineering Context:** Think of chains like CI/CD pipelines. Just as a test engineer chains a build step to an integration test step to a deployment step, an AI engineer uses LCEL to chain a database lookup to an AI reasoning step to an AI formatting step.

---

## 2. Visual Architecture

```mermaid
flowchart TD
    subgraph LCEL_Chain [LCEL Sequential Chain]
        direction TB
        Input[User Input: Topic]
        P1[Prompt 1: Write Blog]
        LLM1[Amazon Nova Pro]
        Parser1[String Parser]
        BlogContent[Output 1: Blog Draft]
        P2[Prompt 2: Generate Title]
        LLM2[Anthropic Claude 3.5 Haiku]
        Parser2[String Parser]
        Final[Output 2: Title]

        Input --> P1
        P1 --> LLM1
        LLM1 --> Parser1
        Parser1 --> BlogContent
        
        BlogContent -.->|"Injected as {blog_text}"| P2
        Input -.->|"Injected as {topic}"| P2
        
        P2 --> LLM2
        LLM2 --> Parser2
        Parser2 --> Final
    end

```

---

## 3. Console Hands-On

**Objective:** Observe how a single chained LangChain execution generates multiple discrete AWS Bedrock API calls by utilizing CloudWatch Logs.

1. Navigate to the **Amazon Bedrock Console** $\rightarrow$ **Settings**.
2. Ensure **Model invocation logging** is enabled and routed to a specific **CloudWatch Log Group** (e.g., `/aws/bedrock/model-invocations`).
3. After executing a multi-step LCEL chain locally or in a Lambda function, navigate to the **Amazon CloudWatch Console** $\rightarrow$ **Log groups**.
4. Open your Bedrock log group.
5. You will see *two distinct* log entries generated fractions of a second apart.
6. Click the first log to inspect the JSON: notice `input.text` contains your Topic prompt. Click the second log: notice `input.text` contains the fully generated blog post from step 1, proving the chain handed off the payload successfully over the AWS network.

---

## 4. Boto3 / LangChain Implementation

This production-grade script modernizes your legacy `LLMChain` example into LCEL, and then implements a **Sequential Chain** (Draft Blog $\rightarrow$ Generate Title) using dictionary mapping (`RunnablePassthrough`).

```python
import logging
import boto3
from botocore.exceptions import ClientError
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("lcel_orchestrator")

class LCELWorkflowManager:
    """Demonstrates modern LangChain Expression Language (LCEL) workflows."""

    def __init__(self, region_name: str = "us-east-1"):
        self.boto3_client = boto3.client("bedrock-runtime", region_name=region_name)
        
        # We can use a powerful model for writing the blog...
        self.writer_llm = ChatBedrockConverse(
            client=self.boto3_client,
            model="anthropic.claude-3-5-sonnet-20241022-v2:0",
            temperature=0.7,
            max_tokens=1000
        )
        
        # ...and a cheaper, faster model just for generating the title
        self.title_llm = ChatBedrockConverse(
            client=self.boto3_client,
            model="amazon.nova-micro-v1:0", 
            temperature=0.9,
            max_tokens=50
        )

    def simple_lcel_chain(self, company: str) -> str:
        """Modernizes the legacy LLMChain example using LCEL."""
        prompt = ChatPromptTemplate.from_messages([
            ("user", "Create a list with the names of the main metrics tracked in the reports of {company}?")
        ])
        
        # LCEL Pipe syntax: Prompt -> LLM -> String Output
        chain = prompt | self.writer_llm | StrOutputParser()
        
        logger.info(f"Invoking simple chain for {company}...")
        return chain.invoke({"company": company})

    def sequential_blog_chain(self, topic: str) -> dict:
        """Chains two LLM tasks: 1. Write Blog -> 2. Write Title."""
        
        # STEP 1: The Blog Writer Chain
        blog_prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert technical blogger."),
            ("user", "Write a 2-paragraph blog post about: {topic}")
        ])
        blog_chain = blog_prompt | self.writer_llm | StrOutputParser()

        # STEP 2: The Title Generator Chain
        title_prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an SEO expert. Write a catchy title for the provided blog."),
            ("user", "Topic: {topic}\n\nBlog Content:\n{blog_content}")
        ])
        title_chain = title_prompt | self.title_llm | StrOutputParser()

        # MASTER CHAIN: Orchestrating the sequence
        # RunnablePassthrough allows us to pass 'topic' down the chain while adding 'blog_content'
        master_chain = (
            {"blog_content": blog_chain, "topic": RunnablePassthrough()}
            | RunnablePassthrough.assign(title=title_chain)
        )

        try:
            logger.info(f"Executing Sequential LCEL Chain for topic: '{topic}'...")
            # Invocation automatically triggers Bedrock twice in sequence
            result = master_chain.invoke({"topic": topic})
            return result
            
        except ClientError as e:
            logger.error(f"Bedrock execution failed: {e.response['Error']['Message']}")
            return {}

if __name__ == "__main__":
    manager = LCELWorkflowManager()
    
    # Run the Sequential Chain
    TOPIC = "The transition from Test Automation to AI Engineering"
    output = manager.sequential_blog_chain(topic=TOPIC)
    
    if output:
        print(f"\n--- TITLE ---\n{output['title']}")
        print(f"\n--- BLOG ---\n{output['blog_content']}\n")

```

---

## 5. Practice Challenges

### Challenge 1: The Parallel Evaluator (`RunnableParallel`)

**Scenario:** You want to compare how Amazon Nova and Anthropic Claude answer the exact same technical question without waiting for one to finish before starting the other.

* **Task:** Use LCEL's `RunnableParallel` to execute two chains simultaneously. Define a single prompt, but route it into a dictionary where `"nova_response"` triggers `amazon.nova-pro-v1:0` and `"claude_response"` triggers `anthropic.claude-3-5-sonnet-20241022-v2:0`. Return the combined JSON.

### Challenge 2: Graceful Fallbacks (`.with_fallbacks()`)

**Scenario:** Amazon Bedrock occasionally returns an `HTTP 429 Too Many Requests` when you hit your provisioned throughput limit.

* **Task:** Modify the `writer_llm` in the LCEL chain. Attach `.with_fallbacks([backup_llm])` so that if Claude 3.5 Sonnet fails or throttles, the chain automatically routes the prompt to Amazon Titan Text Premier to ensure the application never crashes.

---

## 6. Industry Standards & Tips

* **Kill Legacy Classes:** Do not write new code using `LLMChain`, `SequentialChain`, or `ConversationalRetrievalChain`. They are heavy, opaque, and being phased out. LCEL (`|`) is the strict standard for modern AI engineering.
* **Decouple Your Models:** In a sequential chain, do not use a massive, expensive model (like Claude 3.5 Sonnet) for every step. Use a heavy model for deep reasoning (writing the blog, writing code), and chain it into a fast, cheap model (like Nova Micro or Claude Haiku) for lightweight parsing (generating a title, formatting output as JSON). This slashes inference costs.
* **Tracing with LangSmith / AWS X-Ray:** When a 5-step chain fails, identifying which exact LLM call threw the error or hallucinated is notoriously difficult. Instrument your LangChain application with LangSmith or AWS X-Ray to get a visual graph of latency and inputs/outputs at every single pipe (`|`) transition.

---

## 7. Interview Q&A (Targeted for AI Engineers)

### Q1: Why did LangChain move away from `LLMChain` to LangChain Expression Language (LCEL)?

> **Answer:** `LLMChain` hid the execution flow inside a Python class, making it very difficult to stream UI updates, handle async operations, or debug intermediate steps. LCEL exposes the pipeline as a declarative Directed Acyclic Graph (DAG) using the `|` operator. Because every component implements the `Runnable` protocol, LCEL provides streaming (`.stream`), batching (`.batch`), and fallbacks for free, dramatically simplifying production deployments.

### Q2: You need to summarize a 1,000-page log file that drastically exceeds the model's token limit. How do you construct this chain?

> **Answer:** I cannot use a standard "Stuff" chain where the whole document is stuffed into one prompt. I must use a **Map-Reduce** chain architecture. First, I use a Text Splitter to break the log into 2,000-token chunks. The `Map` step runs a summarization chain on each chunk individually (which can be done in parallel). The `Reduce` step takes all those mini-summaries, combines them, and runs a final summarization chain to produce the ultimate output.

### Q3: In a sequential chain where Step 1 generates a SQL query and Step 2 executes and explains it, what happens if Step 1 hallucinates invalid SQL? How do you engineer resilience?

> **Answer:** A naive chain will fail abruptly at Step 2. To engineer resilience in LCEL, we can implement **Self-Correction (Reflection)**. I would route the output of Step 1 into an intermediate validation chain. If the SQL is invalid, the validator chain loops it back to Step 1 with the error message appended, asking it to fix the code. Alternatively, I can attach `.with_fallbacks()` at the component level to gracefully degrade to a different model or return a safe default value.